# ConvNeXt-Tiny v2 — 개선 버전

베이스라인(60.13%) 분석 결과 적용:
- `taper_smooth` Recall 0.22 → **Focal Loss** + **클래스 가중치**로 해결
- 입력 해상도 224 → **320** 으로 증가 (완만한 테이퍼 포착)
- **GradCAM** 시각화로 모델 판단 영역 확인
- **TTA** (Test Time Augmentation) 적용으로 추론 성능 향상

## 0. 패키지 설치 및 공통 설정

In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from tqdm import tqdm

# ── 재현성 고정
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── 경로
DATA_DIR   = './preprocess'
TEST_DIR   = './test_final'
SAVE_DIR   = './checkpoints_v2'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── 하이퍼파라미터
IMG_SIZE    = 320          # 224 → 320 (테이퍼 기울기 포착)
BATCH_SIZE  = 16           # 해상도 증가로 배치 축소
WARMUP_EP   = 5
FINETUNE_EP = 35
LR_HEAD     = 1e-3
LR_FULL     = 5e-5
VAL_RATIO   = 0.2
NUM_WORKERS = 0
PATIENCE    = 10           # Early stopping

CLASSES     = ['mug', 'straight', 'taper_smooth', 'taper_step']
NUM_CLASSES = len(CLASSES)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

print('설정 완료')

## 1. Transform 정의

In [ ]:
NORMALIZE = transforms.Normalize(mean=MEAN, std=STD)

# ── 학습용 (강화된 증강)
train_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),          # 365
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(25, fill=255),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4, fill=255),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.RandomApply([transforms.GaussianBlur(5)], p=0.3),
    # 엣지 강조: taper 윤곽선 학습
    transforms.RandomAdjustSharpness(sharpness_factor=3, p=0.4),
    transforms.ToTensor(),
    NORMALIZE,
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

# ── 검증/테스트용
eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    NORMALIZE,
])

print('Transform 정의 완료')

## 2. 데이터로더 (Stratified Split)

In [ ]:
# Stratified Split
base_ds = datasets.ImageFolder(root=DATA_DIR, transform=eval_tf)
labels  = [s[1] for s in base_ds.samples]

train_idx, val_idx = train_test_split(
    range(len(base_ds)), test_size=VAL_RATIO,
    stratify=labels, random_state=SEED
)

# train에만 train_tf 적용
train_ds_full = datasets.ImageFolder(root=DATA_DIR, transform=train_tf)
train_ds = Subset(train_ds_full, train_idx)
val_ds   = Subset(base_ds,       val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds)}장  |  Val: {len(val_ds)}장')
# 클래스별 분포 확인
from collections import Counter
tr_dist = Counter([labels[i] for i in train_idx])
for idx, cls in enumerate(CLASSES):
    print(f'  {cls}: train {tr_dist[idx]}장')

## 3. Focal Loss 정의

> `taper_smooth` Recall 0.22의 주요 원인: 모델이 쉬운 샘플(mug, straight)에만 집중.  
> Focal Loss는 오분류된 어려운 샘플에 더 큰 가중치를 부여함.

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss = -alpha * (1-pt)^gamma * log(pt)
    gamma=2: 어려운 샘플에 4배 집중
    label_smoothing: 과적합 방지
    """
    def __init__(self, gamma=2.0, label_smoothing=0.1,
                 class_weights=None):
        super().__init__()
        self.gamma = gamma
        self.ls    = label_smoothing
        self.class_weights = class_weights

    def forward(self, inputs, targets):
        ce = F.cross_entropy(
            inputs, targets,
            weight=self.class_weights,
            label_smoothing=self.ls,
            reduction='none'
        )
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


# taper_smooth, straight에 더 높은 가중치
# CLASSES = ['mug', 'straight', 'taper_smooth', 'taper_step']
class_weights = torch.tensor([1.0, 1.5, 2.0, 1.2]).to(DEVICE)

criterion = FocalLoss(gamma=2.0, label_smoothing=0.1,
                      class_weights=class_weights)

print('Focal Loss 정의 완료')
print('클래스 가중치:', dict(zip(CLASSES, class_weights.cpu().tolist())))

## 4. 모델 정의 (ConvNeXt-Tiny)

In [ ]:
torch.manual_seed(SEED)
model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'총 파라미터 수: {total_params/1e6:.1f}M')

## 5. 2-Stage 학습 함수

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            out  = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out  = model(imgs)
        loss = criterion(out, labels)
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


print('학습 함수 정의 완료')

## 6. Stage 1: Warmup (Head만 학습)

In [ ]:
# Feature 고정, classifier만 학습
for name, p in model.named_parameters():
    p.requires_grad = ('classifier' in name)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=1e-4
)
scaler   = GradScaler()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=WARMUP_EP, eta_min=1e-6)

hist = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
best_acc, best_state = 0.0, None
no_improve = 0

print('=== Stage 1: Warmup ===')
print(f"{'Epoch':>5}  {'Tr Loss':>8}  {'Tr Acc':>7}  {'Va Loss':>8}  {'Va Acc':>7}")
print('-' * 47)

for epoch in range(1, WARMUP_EP + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    va_loss, va_acc = evaluate(model, val_loader, criterion)
    scheduler.step()

    hist['train_loss'].append(tr_loss); hist['train_acc'].append(tr_acc)
    hist['val_loss'].append(va_loss);   hist['val_acc'].append(va_acc)

    if va_acc > best_acc:
        best_acc   = va_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1

    print(f"{epoch:>5}  {tr_loss:>8.4f}  {tr_acc*100:>6.2f}%  {va_loss:>8.4f}  {va_acc*100:>6.2f}%")

print(f'\nWarmup 완료 | Best Val Acc: {best_acc*100:.2f}%')

## 7. Stage 2: Finetune (전체 모델 학습)

In [ ]:
# 모든 파라미터 학습 가능
for p in model.parameters():
    p.requires_grad = True

optimizer = optim.AdamW([
    {'params': model.features.parameters(), 'lr': LR_FULL},
    {'params': model.classifier.parameters(), 'lr': LR_FULL * 10},
], weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINETUNE_EP, eta_min=1e-7)
scaler    = GradScaler()
no_improve = 0

print('=== Stage 2: Finetune ===')
print(f"{'Epoch':>5}  {'Tr Loss':>8}  {'Tr Acc':>7}  {'Va Loss':>8}  {'Va Acc':>7}")
print('-' * 47)

for epoch in tqdm(range(1, FINETUNE_EP + 1), desc='Finetune'):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    va_loss, va_acc = evaluate(model, val_loader, criterion)
    scheduler.step()

    hist['train_loss'].append(tr_loss); hist['train_acc'].append(tr_acc)
    hist['val_loss'].append(va_loss);   hist['val_acc'].append(va_acc)

    if va_acc > best_acc:
        best_acc   = va_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch + WARMUP_EP}')
            break

    if epoch % 5 == 0:
        print(f"{epoch+WARMUP_EP:>5}  {tr_loss:>8.4f}  {tr_acc*100:>6.2f}%  {va_loss:>8.4f}  {va_acc*100:>6.2f}%")

save_path = os.path.join(SAVE_DIR, 'best_convnext_v2.pth')
torch.save(best_state, save_path)
print(f'\n최고 Val Acc: {best_acc*100:.2f}%  →  {save_path}')

## 8. 학습 곡선 시각화

In [ ]:
epochs = range(1, len(hist['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ConvNeXt-Tiny v2 학습 곡선', fontsize=13, fontweight='bold')

axes[0].plot(epochs, hist['train_loss'], label='Train', color='#3498DB')
axes[0].plot(epochs, hist['val_loss'],   label='Val',   color='#E74C3C')
axes[0].axvline(WARMUP_EP, color='gray', linestyle='--', alpha=0.5, label='Warmup 끝')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, [v*100 for v in hist['train_acc']], label='Train', color='#3498DB')
axes[1].plot(epochs, [v*100 for v in hist['val_acc']],   label='Val',   color='#E74C3C')
axes[1].axvline(WARMUP_EP, color='gray', linestyle='--', alpha=0.5, label='Warmup 끝')
axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'train_curve_v2.png'), dpi=150)
plt.show()

## 9. Test 평가 (TTA 적용)

> TTA: 동일 이미지를 여러 변환으로 5번 예측 → 평균 → 최종 클래스 결정

In [ ]:
# Best 모델 로드
model.load_state_dict(best_state)
model.eval()

# TTA 변환 목록
tta_transforms = [
    transforms.Compose([transforms.Resize(int(IMG_SIZE*1.14)), transforms.CenterCrop(IMG_SIZE),
                        transforms.ToTensor(), NORMALIZE]),
    transforms.Compose([transforms.Resize(int(IMG_SIZE*1.14)), transforms.CenterCrop(IMG_SIZE),
                        transforms.RandomHorizontalFlip(p=1.0),
                        transforms.ToTensor(), NORMALIZE]),
    transforms.Compose([transforms.Resize(int(IMG_SIZE*1.14)), transforms.CenterCrop(IMG_SIZE),
                        transforms.RandomVerticalFlip(p=1.0),
                        transforms.ToTensor(), NORMALIZE]),
    transforms.Compose([transforms.Resize(int(IMG_SIZE*1.14)),
                        transforms.RandomCrop(IMG_SIZE),
                        transforms.ToTensor(), NORMALIZE]),
    transforms.Compose([transforms.Resize(int(IMG_SIZE*1.14)), transforms.CenterCrop(IMG_SIZE),
                        transforms.RandomAdjustSharpness(3, p=1.0),
                        transforms.ToTensor(), NORMALIZE]),
]

from PIL import Image

def predict_tta(img_path, model, tta_tfs):
    img = Image.open(img_path).convert('RGB')
    probs_list = []
    with torch.no_grad():
        for tf in tta_tfs:
            inp = tf(img).unsqueeze(0).to(DEVICE)
            probs_list.append(torch.softmax(model(inp), dim=1))
    return torch.stack(probs_list).mean(0).argmax(1).item()


# test_final 폴더 평가
test_ds = datasets.ImageFolder(root=TEST_DIR, transform=eval_tf)
test_img_paths = [s[0] for s in test_ds.samples]
test_labels    = [s[1] for s in test_ds.samples]

print(f'테스트 이미지 수: {len(test_ds)}장')

all_preds = []
for path in tqdm(test_img_paths, desc='TTA 추론'):
    all_preds.append(predict_tta(path, model, tta_transforms))

acc = sum(p == t for p, t in zip(all_preds, test_labels)) / len(test_labels)
print(f'\nTest Accuracy (TTA): {acc*100:.2f}%')
print('\n=== Classification Report ===')
print(classification_report(test_labels, all_preds, target_names=CLASSES))

## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(test_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_title('Confusion Matrix — ConvNeXt v2 (TTA)', fontsize=12, fontweight='bold')
ax.set_ylabel('실제'); ax.set_xlabel('예측')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cm_v2.png'), dpi=150)
plt.show()

# taper_smooth vs straight 혼동 수
sm_idx = CLASSES.index('taper_smooth')
st_idx = CLASSES.index('straight')
print(f'\ntaper_smooth → straight 오분류: {cm[sm_idx][st_idx]}건')
print(f'straight → taper_smooth 오분류: {cm[st_idx][sm_idx]}건')

## 11. GradCAM — taper_smooth vs straight 시각화

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0][class_idx] = 1
        output.backward(gradient=one_hot, retain_graph=True)
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[2:],
                            mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx, output.softmax(1).squeeze().detach().cpu().numpy()


gradcam = GradCAM(model, model.features[-2][-1])

# taper_smooth, straight 샘플 각 4장씩 시각화
target_cls = ['taper_smooth', 'straight']
samples_per_cls = 4

fig, axes = plt.subplots(len(target_cls) * samples_per_cls, 3,
                          figsize=(12, 4 * len(target_cls) * samples_per_cls))
fig.suptitle('GradCAM: taper_smooth vs straight', fontsize=13, fontweight='bold')

row = 0
for cls_name in target_cls:
    cls_idx  = CLASSES.index(cls_name)
    cls_paths = [test_img_paths[i] for i, l in enumerate(test_labels) if l == cls_idx]
    for path in cls_paths[:samples_per_cls]:
        img_orig = Image.open(path).convert('RGB')
        img_resized = np.array(img_orig.resize((IMG_SIZE, IMG_SIZE)))
        inp = eval_tf(img_orig).unsqueeze(0).to(DEVICE)
        cam, pred_idx, probs = gradcam.generate(inp)

        heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        overlay = (img_resized * 0.6 + heatmap * 0.4).astype(np.uint8)

        correct = (pred_idx == cls_idx)
        title   = (f'실제: {cls_name}\n예측: {CLASSES[pred_idx]} '
                   f'({probs[pred_idx]*100:.1f}%)')

        axes[row][0].imshow(img_resized)
        axes[row][0].set_title('원본', fontsize=8); axes[row][0].axis('off')
        axes[row][1].imshow(cam, cmap='jet')
        axes[row][1].set_title('GradCAM', fontsize=8); axes[row][1].axis('off')
        axes[row][2].imshow(overlay)
        axes[row][2].set_title(title, fontsize=8,
                                color='green' if correct else 'red')
        axes[row][2].axis('off')
        row += 1

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'gradcam_v2.png'), dpi=120, bbox_inches='tight')
plt.show()
print('GradCAM 저장 완료')

## 12. 최종 결과 요약

In [ ]:
from sklearn.metrics import f1_score

macro_f1  = f1_score(test_labels, all_preds, average='macro')
taper_f1  = f1_score(test_labels, all_preds, average=None)[CLASSES.index('taper_smooth')]

print('┌──────────────────────────────────────────────────┐')
print('│            ConvNeXt-Tiny v2 최종 결과             │')
print('├──────────────────────────────────────────────────┤')
print(f'│  Test Accuracy (TTA) : {acc*100:>6.2f}%                   │')
print(f'│  Macro F1            : {macro_f1:>6.4f}                   │')
print(f'│  taper_smooth F1     : {taper_f1:>6.4f}                   │')
print('├──────────────────────────────────────────────────┤')
print('│  주요 개선 사항                                    │')
print('│  ✅ Focal Loss (gamma=2) + 클래스 가중치           │')
print('│  ✅ 입력 해상도 224 → 320                          │')
print('│  ✅ TTA (5가지 변환 앙상블)                        │')
print('│  ✅ RandomAdjustSharpness (윤곽선 강조)            │')
print('│  ✅ Early Stopping (patience=10)                  │')
print('└──────────────────────────────────────────────────┘')